In [1]:
from dtgraph import Neo4jGraph, Rule, Transformation

hostname = "localhost"
password = "internship"
uri = f"bolt://{hostname}:7687"

graph = Neo4jGraph(uri, database="neo4j", username="neo4j", password=password)

In [2]:
import sys
import os

# Go to project root
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Add dtgraph folder to path
sys.path.append(os.path.join(project_root, "../dtgraph"))

In [3]:
from type_checking.environment import Environment

env = Environment("../../dtgraph/type_checking/ENVs/env_fraud.json")

##### Rules

In [6]:
Rule1 = Rule('''
MATCH (c:Client)
WHERE NOT c:Mule
WHERE c.name NOT null
OPTIONAL MATCH (c)-[:HAS_EMAIL]->(e:Email)
OPTIONAL MATCH (c)-[:HAS_PHONE]->(p:Phone)
OPTIONAL MATCH (c)-[:HAS_SSN]->(s:SSN)
WITH 
    c,
    collect(DISTINCT e.email) AS emails,
    collect(DISTINCT p.phoneNumber) AS phones,
    collect(DISTINCT s.ssn) AS ssns
LIMIT 2000
GENERATE
(p = (c.id):Person {
    id = c.id,
    name = c.name,
    name_camel_case = apoc.text.upperCamelCase(c.name),
    email = head(emails),
    phone = head(phones),
    ssn = head(ssns),
    names = c.name,
    namess = c.name,
    namesss = c.name,
})
''', env=env, type_strict=False)

Rule2 = Rule('''
MATCH (c:Client:Mule)
OPTIONAL MATCH (c)-[:HAS_EMAIL]->(e:Email)
OPTIONAL MATCH (c)-[:HAS_PHONE]->(p:Phone)
OPTIONAL MATCH (c)-[:HAS_SSN]->(s:SSN)
WITH 
    c,
    collect(DISTINCT e.email) AS emails,
    collect(DISTINCT p.phoneNumber) AS phones,
    collect(DISTINCT s.ssn) AS ssns
LIMIT 2000
GENERATE
(p = (c.id):Scammer {
    id = c.id,
    name = c.name,
    name_camel_case = apoc.text.upperCamelCase(c.name),
    email = head(emails),
    phone = head(phones),
    ssn = head(ssns)
})
''', env=env, type_strict=True)

Rule3 = Rule('''
MATCH (c:Client)-[:PERFORMED]->(t:CashIn)
WITH c, t LIMIT 2000
GENERATE
(p = (c.id):)-[():PERFORMED_CASHIN]->((t.id):CashIn {
        original_amount = t.amount,
        formatted_amount = round(t.amount * 100) / 100.0,
        is_large_amount = t.amount > 150000
    })
''', env=env, type_strict=True)

Rule4 = Rule('''
MATCH (c:Client)-[:PERFORMED]->(t:CashOut)
WITH c, t LIMIT 2000
GENERATE
(p = (c.id):)-[():PERFORMED_CASHOUT]->(tx = (t.id):CashOut {
        original_amount = t.amount,
        formatted_amount = round(t.amount * 100) / 100.0,
        is_large_amount = t.amount > 150000
    })
''', env=env, type_strict=True)

Rule5 = Rule('''
MATCH (c:Client)-[:PERFORMED]->(t:Payment)
WITH c, t LIMIT 2000
GENERATE
(p = (c.id):)-[():PERFORMED_PAYMENT]->(tx = (t.id):Payment)
''', env=env, type_strict=True)

Rule6 = Rule('''
MATCH (c:Client)-[:PERFORMED]->(t:Transfer)
WITH c, t LIMIT 2000
GENERATE
(p = (c.id):)-[():PERFORMED_TRANSFER]->(tx = (t.id):Transfer)
''', env=env, type_strict=True)

Rule7 = Rule('''
MATCH (c:Client)-[:PERFORMED]->(t:Debit)
WITH c, t LIMIT 2000
GENERATE
(p = (c.id):)-[():PERFORMED_DEBITS]->(tx = (t.id):Debit)
''', env=env, type_strict=True)

In [ ]:
from dtgraph.type_checking.check_types import check_types

check_types([Rule1], env)

##### Applying Rules

In [9]:
my_transform = Transformation([Rule7])
my_transform.apply_on(graph)

Index: Added 0 index, completed after 29 ms.
Before LHS: MATCH (c:Client)-[:PERFORMED]->(t:Debit)
WITH c, t LIMIT 2000
props: set()
After LHS: MATCH (c:Client)-[:PERFORMED]->(t:Debit)
WITH c, t LIMIT 2000


Rule: Added 4820 labels, created 2820 nodes, set 4820 properties, created 2000 relationships, completed after 3524 ms.


3524

##### Abort Transformation

In [10]:
my_transform.abort()

Index: Removed 1 index, completed after 51 ms.
Abort: Deleted 2820 nodes, deleted 2000 relationships, completed after 364 ms.
